# Taller: monta tú un modelo de marketing mix

Segunda parte del seminario. En la primera hora has visto qué es una priori, qué es una
posteriori y por qué un modelo jerárquico presta información entre grupos. Aquí no vas a
repetir aquello con otros datos: vas a **elegir prioris y defenderlas**, que es la parte
de la que nadie habla, y a **diagnosticar** lo que salga.

El recorrido son dos modelos sobre los mismos datos:

1. Los datos y la regresión que miente.
2. El esqueleto: nivel y controles. Tres prioris que decidir.
3. Ajuste y diagnóstico del esqueleto.
4. Ahora los medios: adstock y una priori sobre el ROI.
5. Ajuste y diagnóstico del modelo completo.
6. La pregunta que le importa a alguien.

Seis ejercicios: tres de prioris, dos de diagnóstico y uno de decisión. Y ya está, que son
las dos de la tarde.

**Esta es la versión resuelta.** El enunciado es el mismo que el de `taller.ipynb`; lo que
cambia es que las celdas de los ejercicios vienen con código y con la respuesta escrita
debajo. Son seis: tres de prioris, dos de diagnóstico y uno de decisión.

Dos avisos. Los números concretos salen de `SEMILLA = 42`: si la cambias, las cifras de las
respuestas se mueven en el último decimal, no las conclusiones. Y si no quieres esperar al
muestreo del modelo completo, en la sección 5 hay un `MUESTREAR = False` que carga la
posteriori guardada.

---

**Cómo ejecutar esto.** En Google Colab no hay que instalar nada: la primera celda de
código se encarga de las dependencias y de traerse los datos. Ejecuta de arriba abajo. Si
Colab te pide reiniciar el entorno después de instalar, reinícialo y vuelve a empezar por
la primera celda. En local, el entorno de `requirements.txt` y a correr.

In [ ]:
# En Colab: instala lo que falta y trae datos y código del repositorio.
# En local no hace nada: se da por hecho el entorno de requirements.txt.
from pathlib import Path

try:
    import google.colab  # noqa: F401

    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    %pip install -q "numpy>=1.26,<2.5" "pymc>=5.15,<6" "arviz>=0.23,<1" h5netcdf
    if not Path("/content/seminar-bayes").exists():
        !git clone -q --depth 1 https://github.com/lhansa/seminar-bayes.git /content/seminar-bayes
    %cd /content/seminar-bayes/notebooks

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pymc as pm
import pytensor.tensor as pt
import arviz as az
import statsmodels.api as sm

from src.estilo import ACENTO, GRISES, aplicar_estilo
from src.mmm import (
    CANALES,
    CONTROLES,
    MAX_LAG,
    cargar_datos,
    escalar_controles,
    escalar_kpi,
    escalar_medios,
    matriz_rezagos,
    pesos_adstock,
    vida_media,
)

aplicar_estilo()
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los datos y la regresión que miente

Un *marketing mix model* responde a una pregunta vieja: de todo lo que vendo, ¿cuánto se
debe a lo que me gasto en publicidad? Antes se hacía con una regresión y una hoja de
cálculo. Ahora se hace con una regresión y un modelo bayesiano, que es lo mismo pero
admitiendo en voz alta que los datos no dan para tanto.

Usamos los datos simulados de **Google Meridian**, la librería de MMM de Google: 156
semanas, cinco canales de pago con impresiones y gasto, dos controles y un KPI, los
ingresos.

Esta sección va entera de mirar. No hay ejercicio; el trabajo empieza en la siguiente.

In [ ]:
df = cargar_datos(RAIZ)
print(f"{len(df)} semanas, de {df['time'].min():%Y-%m-%d} a {df['time'].max():%Y-%m-%d}")
df.head()

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ejes[0].plot(df["time"], df["revenue"] / 1e6, color=ACENTO)
ejes[0].set_ylabel("Ingresos (M€)")

for canal, color in zip(CANALES, GRISES + [ACENTO]):
    ejes[1].plot(df["time"], df[f"{canal}_spend"] / 1e3, color=color, label=canal, lw=1)
ejes[1].set_ylabel("Inversión (k€)")
ejes[1].legend(ncol=5, fontsize=9)

fig.tight_layout()
plt.show()

El modelo que ajusta todo el mundo: ingresos contra inversión de cada canal, más los dos
controles. Mínimos cuadrados. El coeficiente de cada canal se lee directamente como
**ROI**: euros de ingreso incremental por euro invertido.

In [ ]:
X = df[[f"{c}_spend" for c in CANALES] + CONTROLES]
ols = sm.OLS(df["revenue"], sm.add_constant(X)).fit()

filas = [f"{c}_spend" for c in CANALES]
resumen_ols = pd.DataFrame({
    "roi_ols": ols.params[filas].values,
    "ic_bajo": ols.conf_int().loc[filas, 0].values,
    "ic_alto": ols.conf_int().loc[filas, 1].values,
    "p_valor": ols.pvalues[filas].values,
    "corr_con_ingresos": [df[f].corr(df["revenue"]) for f in filas],
}, index=CANALES)

print(f"R² = {ols.rsquared:.3f}")
resumen_ols.round(2)

In [ ]:
correlaciones = df[[f"{c}_spend" for c in CANALES]].corr()
correlaciones.index = correlaciones.columns = CANALES
correlaciones.round(2)

Un $R^2$ de 0,24 y ningún canal significativo. Los intervalos de confianza van de perder
dinero a doblarlo, todos. Tres de los cinco canales correlacionan **negativamente** con la
facturación. Y los canales 2, 3 y 4 se mueven juntos, entre 0,62 y 0,70: no hay modelo,
bayesiano ni de los otros, que separe limpiamente lo que siempre sube y baja a la vez.

Leído en frío, esto dice que la publicidad no sirve para nada. También podría estar
diciendo que la empresa invierte más justo cuando las cosas van peor. Con 156 semanas las
dos explicaciones caben igual de bien.

La salida fácil es pedir más datos. No los va a haber: la inversión la decide un equipo de
marketing, no un experimento aleatorizado, y lleva años decidiéndola igual. La otra salida
es escribir en el modelo lo que ya sabes del negocio.

Y para eso no empezamos por la publicidad. Empezamos por el modelo sin publicidad.

## 2. El esqueleto: nivel y controles

Quita los medios y queda esto:

$$y_t \sim \mathcal{N}\!\left(\mu + \sum_c \gamma_c z_{tc},\ \sigma^2\right)$$

Un nivel, dos controles y un error. Tres parámetros, tres prioris que elegir. Parece poca
cosa para un taller de marketing mix y es justo lo que hace falta: con tres prioris se ve
el oficio entero, y este modelo ajusta en segundos, así que puedes equivocarte varias veces
antes de que se acabe la sesión.

**Antes, el escalado.** Los ingresos están en millones. Si metes eso tal cual, cualquier
priori que escribas es un disparate: `Normal(0, 5)` sobre una variable que vale 8.000.000
no es una priori poco informativa, es una priori absurdamente informativa a favor del cero.

Así que, como Meridian: al KPI se le resta la media y se divide por su desviación típica, y
los controles se dejan con media cero y desviación típica uno. **Todas las prioris del
taller están escritas en esta escala.** Si cambias el escalado, cambias el modelo.

In [ ]:
y = df["revenue"].to_numpy()
z = df[CONTROLES].to_numpy(float)

y_esc, y_media, y_sd = escalar_kpi(y)
z_esc = escalar_controles(z)

print(f"ingresos: media {y_media / 1e6:.2f} M€, desviación típica {y_sd / 1e6:.2f} M€")
print(f"y_esc:    media {y_esc.mean():.2f}, desviación típica {y_esc.std():.2f}")

### Tres parámetros, tres decisiones

| Parámetro | Meridian | Aquí | Qué dice la de aquí |
|---|---|---|---|
| $\mu$ | $\mathcal{N}(0, 5)$ | $\mathcal{N}(0, 1)$ | el KPI está estandarizado: su nivel es cero y no se va lejos |
| $\gamma_c$ | $\mathcal{N}(0, 5)$ | $\mathcal{N}(0, 1)$ | un control que se mueve una desviación típica mueve el KPI, como mucho, otra |
| $\sigma$ | $\text{HalfNormal}(5)$ | $\text{Exp}(1)$ | el error no puede ser mayor que la desviación típica de lo que quieres explicar |

Las de Meridian no son un error: su modelo es jerárquico por regiones y tiene miles de
observaciones repartidas por geografías, así que los datos las tapan. Aquí hay 156 semanas
y una sola serie. Aquí no las tapan.

Pero esto es una tabla, y una tabla no demuestra nada. Una priori no se juzga mirando su
fórmula: se juzga mirando **qué datos genera**. Abajo está el modelo escrito de forma que
puedas cambiarle las prioris sin reescribirlo, y una función que simula 500 series de 156
semanas desde la priori, sin haber visto ni un dato.

In [ ]:
coords = {"control": CONTROLES, "semana": df["time"].dt.date.astype(str)}


def construir_esqueleto(mu_sd=1.0, gamma_sd=1.0, sigma_prior="exp"):
    """Modelo de nivel y controles, con las prioris como argumentos."""
    with pm.Model(coords=coords) as modelo:
        mu = pm.Normal("mu", 0.0, mu_sd)
        gamma = pm.Normal("gamma", 0.0, gamma_sd, dims="control")
        if sigma_prior == "exp":
            sigma = pm.Exponential("sigma", 1.0)
        else:
            sigma = pm.HalfNormal("sigma", 5.0)

        media = mu + pt.dot(z_esc, gamma)
        pm.Normal("y", mu=media, sigma=sigma, observed=y_esc, dims="semana")
    return modelo


def ingresos_previos(modelo, draws=500):
    """Simula series de ingresos desde la priori y las devuelve en euros."""
    with modelo:
        previa = pm.sample_prior_predictive(draws=draws, random_seed=SEMILLA)
    return previa.prior_predictive["y"].values.reshape(-1, len(df)) * y_sd + y_media


esqueleto = construir_esqueleto()
esqueleto

In [ ]:
y_previo = ingresos_previos(esqueleto)

fig, eje = plt.subplots(figsize=(7.5, 3.4))
eje.hist(y_previo.ravel() / 1e6, bins=80, color="#AAAAAA", density=True,
         label="simulado desde la priori")
eje.hist(y / 1e6, bins=30, color=ACENTO, density=True, alpha=0.8, label="observado")
eje.set_xlabel("Ingresos semanales (M€)")
eje.legend(fontsize=9)
fig.tight_layout()
plt.show()

print(f"semanas simuladas con ingresos negativos: {100 * (y_previo < 0).mean():.1f} %")

### Ejercicio 1: elige las prioris del esqueleto

Arriba están las apretadas. Ahora prueba las otras y decide con qué te quedas.

- Simula desde las prioris de Meridian: `construir_esqueleto(mu_sd=5, gamma_sd=5,
  sigma_prior="halfnormal")`. ¿Qué porcentaje de las semanas simuladas sale con ingresos
  **negativos**? Vender menos que nada.
- Cambia una priori cada vez, dejando las otras dos apretadas. ¿De cuál de las tres es la
  culpa?
- `mu ~ Normal(0, 5)` sobre un KPI estandarizado: escribe en una frase, con la palabra
  "millones" dentro, qué está diciendo exactamente sobre la facturación de una semana.

Una priori que genera ingresos negativos no es "poco informativa". Es falsa. Y arreglarla
ahora es gratis; después del ajuste, ya no.

In [ ]:
# Las prioris de Meridian, tal cual.
meridian = construir_esqueleto(mu_sd=5.0, gamma_sd=5.0, sigma_prior="halfnormal")
y_meridian = ingresos_previos(meridian)
print(f"Meridian: {100 * (y_meridian < 0).mean():.1f} % de semanas con ingresos negativos\n")

# Una priori ancha cada vez, las otras dos apretadas: así se ve de quién es la culpa.
pruebas = {
    "las tres apretadas": {},
    "solo mu ~ N(0, 5)": {"mu_sd": 5.0},
    "solo gamma ~ N(0, 5)": {"gamma_sd": 5.0},
    "solo sigma ~ HalfNormal(5)": {"sigma_prior": "halfnormal"},
}
for nombre, kwargs in pruebas.items():
    simulado = ingresos_previos(construir_esqueleto(**kwargs))
    print(f"{nombre:28s} {100 * (simulado < 0).mean():5.1f} % negativas, "
          f"sd {simulado.std() / 1e6:6.2f} M€")

**Respuesta.** Con las prioris de Meridian, el **6,7 %** de las semanas simuladas sale con
ingresos negativos. Una empresa que vende menos que nada diez semanas de cada 156, y el
modelo se lo cree antes de mirar un solo dato.

Repartiendo la culpa, con las otras dos apretadas: `gamma ~ Normal(0, 5)` sola deja un
2,8 % de semanas negativas, `sigma ~ HalfNormal(5)` un 1,0 % y `mu ~ Normal(0, 5)` un 0,2 %.
La peor con diferencia es la de los controles, y se entiende: está diciendo que un control
que se mueve una desviación típica puede mover el KPI **cinco**, y hay dos controles, así
que entre los dos se lo llevan diez veces. La desviación típica de los ingresos simulados
pasa de 1,33 M€ con las tres apretadas a 4,25 M€ con esa sola.

Y `mu ~ Normal(0, 5)` sobre un KPI estandarizado, dicho en cristiano: "la facturación de una
semana normal está entre 2,7 y 14,2 millones, y cualquier valor de ahí en medio me parece
igual de creíble". La facturación real de las 156 semanas va de 6,8 a 9,8 millones. Nadie
firmaría esa frase en una reunión; escrita como priori, se firma sin leerla.

## 3. Ajuste y diagnóstico del esqueleto

Ahora sí, que vea los datos.

Con el resultado delante hay que separar dos preguntas que se confunden todo el rato:

- **¿Ha funcionado el muestreador?** `r_hat`, `ess_bulk`, `ess_tail`, divergencias.
- **¿Describe el modelo los datos?** La predictiva posterior.

Un modelo puede muestrear de maravilla y ser una tontería. Y al revés. Este de aquí es un
sitio estupendo para verlo, porque va a hacer exactamente una de las dos cosas.

In [ ]:
with esqueleto:
    idata_esqueleto = pm.sample(draws=1000, tune=1000, chains=4, random_seed=SEMILLA)
    idata_esqueleto.extend(pm.sample_posterior_predictive(idata_esqueleto, random_seed=SEMILLA))

resumen_esqueleto = az.summary(idata_esqueleto, var_names=["mu", "gamma", "sigma"])
print("divergencias:", int(idata_esqueleto.sample_stats["diverging"].sum()))
resumen_esqueleto.round(2)

In [ ]:
az.plot_ppc(idata_esqueleto, num_pp_samples=100, colors=[ACENTO, "#7A7A7A", "black"])
plt.show()

### Ejercicio 2: diagnostica el esqueleto

- Mira `r_hat`, `ess_bulk` y las divergencias. ¿Ha funcionado el muestreador?
- Dibuja la posteriori de cada `gamma` contra su priori `Normal(0, 1)`. Uno de los dos
  controles ha movido al modelo y el otro te está devolviendo la priori con otro nombre.
  ¿Cuál es cuál, y qué haces con el que sobra?
- `y_esc` tiene desviación típica uno por construcción. Con el `sigma` que ha salido,
  ¿qué proporción de la varianza explica el modelo?
- Con eso en la mano: ¿qué preguntas responde este modelo y cuáles no?

La trampa de la sección es esa: el muestreador va a salir impecable. Eso no dice
absolutamente nada sobre si el modelo sirve para algo.

In [ ]:
from scipy import stats

# ¿Ha funcionado el muestreador?
print(resumen_esqueleto[["r_hat", "ess_bulk", "ess_tail"]].round(2))

# Priori contra posteriori de cada control.
rejilla = np.linspace(-3, 3, 300)
fig, ejes = plt.subplots(1, len(CONTROLES), figsize=(10, 3.2), sharey=True)
for i, (eje, control) in enumerate(zip(ejes, CONTROLES)):
    posterior = idata_esqueleto.posterior["gamma"].values[..., i].ravel()
    eje.plot(rejilla, stats.norm(0, 1).pdf(rejilla), color="#AAAAAA", lw=2, label="priori")
    eje.hist(posterior, bins=40, density=True, color=ACENTO, alpha=0.85, label="posteriori")
    eje.set_title(control, fontsize=10)
    eje.set_xlabel("γ")
ejes[0].legend(fontsize=9)
fig.tight_layout()
plt.show()

# y_esc tiene varianza uno, así que sigma² es la parte que el modelo no explica.
sigma_esqueleto = float(idata_esqueleto.posterior["sigma"].mean())
print(f"sigma = {sigma_esqueleto:.2f}  ->  varianza explicada = {1 - sigma_esqueleto ** 2:.1%}")

**Respuesta.** El muestreador, impecable: `r_hat` 1,00 en los cuatro parámetros, `ess_bulk`
entre 4.300 y 5.800 sobre 4.000 muestras, cero divergencias. Tres segundos.

El modelo, flojo. `sigma` sale en 0,93 y, como `y_esc` tiene varianza uno por construcción,
eso deja la varianza explicada en el **12,7 %**. Casi el 90 % de lo que pasa en la
facturación semanal sigue sin explicación.

De los dos controles habla uno solo. `competitor_sales_control` se planta en −0,42 con una
posteriori estrecha y claramente desplazada de la priori: cuando la competencia vende, se
nota. `sentiment_score_control` sale en 0,10, con el cero dentro del intervalo y una
anchura parecida a la de su priori; ahí el modelo no ha aprendido nada. No hace falta
quitarlo —no estorba, y su priori ya lo está sujetando—, pero no vayas a la reunión a
contar que el sentimiento de marca mueve las ventas, porque eso lo has escrito tú.

Y lo que responde este modelo es "¿cuánto se explica con la competencia?". Nada más. Para la
pregunta que nos ha traído aquí —cuánto aporta cada canal— no sirve: no hay canales
dentro.

## 4. Ahora los medios

El esqueleto explica poco y, de lo poco que explica, nada es publicidad. Toca meterla.

Pero el gasto no entra tal cual, porque la regresión de la sección 1 asume dos cosas que
son falsas:

1. Que el anuncio de esta semana no vende nada la semana que viene.
2. Que el euro un millón hace lo mismo que el primero.

Hoy arreglamos la primera. **Adstock**: la inversión de una semana se reparte en las
siguientes,

$$\tilde{x}_{t} = \frac{\sum_{l=0}^{L} \alpha^{l}\, x_{t-l}}{\sum_{l=0}^{L} \alpha^{l}}, \qquad \alpha \in [0, 1]$$

Los pesos se normalizan a propósito: $\alpha$ **reparte** el efecto en el tiempo, no lo
infla. Meridian usa $L = 8$ semanas.

La segunda —la saturación, que el rendimiento por euro caiga según subes la inversión— se
modela con una curva de Hill y mete un parámetro más por canal. Hoy se queda fuera: son
diez minutos de explicación que no tienen nada de bayesiana, y el taller va de otra cosa.
La función `hill()` está en `src/mmm.py` por si quieres añadirla luego.

**El escalado de los medios** sí es distinto del KPI: cada canal se divide por la **mediana
de sus semanas con inversión**. Una unidad = "una semana normal de ese canal". Ni media ni
máximo: mediana de lo positivo, porque hay canales con semanas a cero y una semana sin
invertir no dice nada sobre la escala del canal.

In [ ]:
fig, eje = plt.subplots(figsize=(7.5, 3.4))

lags = np.arange(MAX_LAG + 1)
for alpha_, color in zip([0.1, 0.4, 0.7, 0.9], GRISES[:3] + [ACENTO]):
    w = pesos_adstock(np.array([alpha_]), MAX_LAG)[0]
    eje.plot(lags, w, "o-", color=color,
             label=f"α = {alpha_} (vida media {vida_media(alpha_):.1f} sem.)")
eje.set_xlabel("semanas desde la impresión")
eje.set_ylabel("peso")
eje.legend(fontsize=9)

fig.tight_layout()
plt.show()

In [ ]:
x = df[[f"{c}_impression" for c in CANALES]].to_numpy(float)
gasto = df[[f"{c}_spend" for c in CANALES]].to_numpy(float)

x_esc, escala_medios = escalar_medios(x)
gasto_total = gasto.sum(axis=0)

pd.DataFrame({
    "escala (mediana impresiones > 0)": escala_medios,
    "inversión total (M€)": gasto_total / 1e6,
    "semanas a cero": (x == 0).sum(axis=0),
}, index=CANALES).round(2)

### $\beta$ no lleva priori

El modelo completo es el esqueleto más un término de medios:

$$y_t \sim \mathcal{N}\!\left(\mu + \sum_m \beta_m \tilde{x}_{tm} + \sum_c \gamma_c z_{tc},\ \sigma^2\right)$$

Quedan por decidir $\alpha_m$ y $\beta_m$.

La de $\alpha$ es fácil de defender: $\mathcal{U}(0, 1)$. Ni idea de cuánto dura el efecto
de un anuncio, y no pasa nada por decirlo.

¿Y $\beta_2$? Es el coeficiente de una inversión rezagada, dividida por su mediana, sobre
un KPI estandarizado. Nadie tiene una intuición sobre eso. Nadie.

Sobre lo que sí tiene intuición el equipo de marketing es sobre el **ROI**: cuánto ingreso
incremental deja un euro invertido en ese canal. Así que la priori se pone ahí:

$$\text{roi}_m \sim \text{LogNormal}(0{,}2;\ 0{,}9)$$

y $\beta_m$ se **deduce**. El ingreso incremental del canal $m$ es lo que se pierde si lo
apagas del todo, y por construcción tiene que valer $\text{roi}_m$ por lo invertido:

$$\beta_m \cdot \sum_t \tilde{x}_{tm} \cdot sd(y) = \text{roi}_m \cdot \text{gasto}_m
\quad\Longrightarrow\quad
\beta_m = \frac{\text{roi}_m \cdot \text{gasto}_m}{sd(y) \cdot \sum_t \tilde{x}_{tm}}$$

$\beta_m$ deja de ser un parámetro con vida propia: es una cuenta que depende del ROI y del
adstock. Esto es lo mejor que hace Meridian, y no tiene nada que ver con la programación
probabilística: es reconocer que **la priori hay que escribirla en las unidades en las que
la gente sabe pensar**.

### Ejercicio 3: ¿qué dice esa priori sobre el ROI?

Traduce $\text{LogNormal}(0{,}2;\ 0{,}9)$ a algo que puedas decir en una reunión sin que se
te quede nadie mirando:

- ¿Cuál es el ROI mediano que asume?
- ¿Entre qué dos valores está el 90 % central?
- ¿Qué probabilidad le da a que un canal pierda dinero (ROI < 1)?

Pista: `scipy.stats.lognorm(s=0.9, scale=np.exp(0.2))`.

Y la pregunta incómoda: esa misma priori se le pone a los cinco canales. ¿Te parece
razonable, sabiendo que uno se lleva siete veces más presupuesto que otro?

In [ ]:
from scipy import stats

priori_roi = stats.lognorm(s=0.9, scale=np.exp(0.2))

print(f"ROI mediano:  {priori_roi.median():.2f}")
print(f"90 % central: [{priori_roi.ppf(0.05):.2f}, {priori_roi.ppf(0.95):.2f}]")
print(f"P(ROI < 1):   {priori_roi.cdf(1):.1%}")

print("\ninversión total por canal (M€):")
print(pd.Series(gasto_total / 1e6, index=CANALES).round(1))

**Respuesta.** ROI mediano **1,22**: por cada euro invertido, 1,22 euros de ingreso
incremental. El 90 % central va de **0,28 a 5,37**. Y le da un **41 %** de probabilidad a
que un canal esté perdiendo dinero.

En la reunión suena así: "asumo que la publicidad devuelve algo más de lo que cuesta, pero
admito casi cualquier cosa entre perder tres cuartas partes de lo invertido y quintuplicarlo,
y me parece casi igual de probable perder que ganar". Es una priori honesta para alguien que
no tiene ni idea. También es enorme.

Y no, no es razonable ponérsela igual a los cinco canales. `Channel3` se lleva 87,9 M€ y
`Channel2` 12,1 M€, siete veces menos. Del canal grande hay muchísima más información
—histórico, experimentos, referencias del sector— y la priori debería notarlo siendo más
estrecha. Que Meridian ponga la misma por defecto no es una recomendación: es lo único que
puede hacer una librería que no sabe de qué empresa le estás hablando.

### Una priori no es de un parámetro: es del modelo entero

Falta una cosa antes de escribirlo, y es la trampa del taller.

En el esqueleto apretaste $\mu$ a $\mathcal{N}(0, 1)$, y era lo correcto: el KPI está
estandarizado, su nivel es cero. Pero en el modelo completo $\mu$ ya no es el nivel de los
ingresos: es lo que queda **después** de restar los medios. Y el término de medios, con la
priori del ROI que acabas de mirar, no es pequeño.

Así que $\mu$ vuelve a $\mathcal{N}(0, 5)$. No por prudencia ni por simetría: porque tiene
que poder irse a $-3$ o $-4$ para compensar lo que aportan los medios. Si la dejas apretada, el
modelo no tiene forma de bajar el nivel base y lo paga el muestreador, semanas después,
cuando ya nadie se acuerda de esta celda.

Cuánto aportan exactamente los medios lo mides tú en el ejercicio 4.

In [ ]:
rezagos = matriz_rezagos(x_esc, MAX_LAG)  # (semanas, rezagos, canales)

coords_completo = dict(coords, canal=CANALES)

with pm.Model(coords=coords_completo) as completo:
    alpha = pm.Uniform("alpha", 0.0, 1.0, dims="canal")
    roi = pm.LogNormal("roi", mu=0.2, sigma=0.9, dims="canal")
    gamma = pm.Normal("gamma", 0.0, 1.0, dims="control")
    mu = pm.Normal("mu", 0.0, 5.0)  # la ancha a propósito: compensa la contribución de medios
    sigma = pm.Exponential("sigma", 1.0)

    # Adstock: media ponderada de los 9 rezagos, con pesos que suman uno.
    pesos = pesos_adstock(alpha, MAX_LAG)                    # (canales, rezagos)
    adstock = pt.sum(rezagos * pesos.T[None, :, :], axis=1)  # (semanas, canales)

    # Beta no es un parámetro libre: sale del ROI.
    beta = pm.Deterministic(
        "beta", roi * gasto_total / (y_sd * pt.sum(adstock, axis=0)), dims="canal"
    )

    contribucion = pm.Deterministic("contribucion", roi * gasto_total, dims="canal")

    media = mu + pt.dot(adstock, beta) + pt.dot(z_esc, gamma)
    pm.Normal("y", mu=media, sigma=sigma, observed=y_esc, dims="semana")

completo

In [ ]:
with completo:
    previa = pm.sample_prior_predictive(draws=500, random_seed=SEMILLA)

y_previo_completo = previa.prior_predictive["y"].values.reshape(-1, len(df)) * y_sd + y_media
contrib_previa = previa.prior["contribucion"].values.reshape(-1, len(CANALES)).sum(axis=1)

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))

ejes[0].hist(y_previo_completo.ravel() / 1e6, bins=80, color="#AAAAAA", density=True,
             label="simulado desde la priori")
ejes[0].hist(y / 1e6, bins=30, color=ACENTO, density=True, alpha=0.8, label="observado")
ejes[0].set_xlabel("Ingresos semanales (M€)")
ejes[0].legend(fontsize=9)

ejes[1].hist(100 * contrib_previa / y.sum(), bins=80, color=ACENTO)
ejes[1].set_xlabel("% de los ingresos atribuido a los medios, según la priori")

fig.tight_layout()
plt.show()

### Ejercicio 4: ¿cuánta publicidad hay metida en tu priori?

El gráfico de la izquierda es el de siempre: ¿considera el modelo normales unos ingresos
imposibles? El de la derecha es el que de verdad importa: cuánto de la facturación le
atribuye tu modelo a la publicidad **antes de ver un solo dato**.

- Calcula la mediana y el percentil 95 de ese porcentaje.
- Compara la mediana de los ingresos que simula el modelo con la mediana observada, 8,5 M€.
  Antes de ver un solo dato ya estás facturando de más. ¿Cuántas desviaciones típicas de
  más? Ahí tienes el hueco que `mu` va a tener que tapar, y por eso ha vuelto a abrirse.
- Tu jefe te dice que la publicidad explica como mucho el 30 % de la facturación. Busca una
  `LogNormal` sobre el ROI que deje el percentil 95 de ese porcentaje por debajo de 30.
  Pruébala y vuelve a simular hasta que cuadre.

Esta es la pregunta que hay que hacerle a cualquier MMM, propio o de proveedor. Si no saben
responderla, el número que te van a dar es el suyo, no el de los datos.

In [ ]:
pct_previo = 100 * contrib_previa / y.sum()
print(f"contribución de medios según la priori: mediana {np.median(pct_previo):.1f} %, "
      f"p95 {np.percentile(pct_previo, 95):.1f} %")


def simular_completo(mu_sd=5.0, roi_mu=0.2, roi_sigma=0.9, draws=500):
    """El modelo completo otra vez, cambiando la priori de mu y la del ROI."""
    with pm.Model(coords=coords_completo):
        alpha_ = pm.Uniform("alpha", 0.0, 1.0, dims="canal")
        roi_ = pm.LogNormal("roi", mu=roi_mu, sigma=roi_sigma, dims="canal")
        gamma_ = pm.Normal("gamma", 0.0, 1.0, dims="control")
        mu_ = pm.Normal("mu", 0.0, mu_sd)
        sigma_ = pm.Exponential("sigma", 1.0)

        adstock_ = pt.sum(rezagos * pesos_adstock(alpha_, MAX_LAG).T[None, :, :], axis=1)
        beta_ = roi_ * gasto_total / (y_sd * pt.sum(adstock_, axis=0))
        pm.Deterministic("contribucion", roi_ * gasto_total, dims="canal")
        pm.Normal("y", mu=mu_ + pt.dot(adstock_, beta_) + pt.dot(z_esc, gamma_),
                  sigma=sigma_, observed=y_esc, dims="semana")

        previa_ = pm.sample_prior_predictive(draws=draws, random_seed=SEMILLA)

    ingresos = previa_.prior_predictive["y"].values.reshape(-1, len(df)) * y_sd + y_media
    aporte = previa_.prior["contribucion"].values.reshape(-1, len(CANALES)).sum(axis=1)
    return ingresos, 100 * aporte / y.sum()


# El hueco que va a tener que tapar mu, medido en desviaciones típicas del KPI.
hueco = np.median(y_previo_completo) - np.median(y)
print(f"\nmediana simulada  {np.median(y_previo_completo) / 1e6:5.2f} M€")
print(f"mediana observada {np.median(y) / 1e6:5.2f} M€")
print(f"de más: {hueco / 1e6:.2f} M€ = {hueco / y_sd:.1f} desviaciones típicas")

# Y la trampa: apretar mu no se nota en la simulación previa.
ingresos_mu1, _ = simular_completo(mu_sd=1.0)
print(f"\ncon mu ~ N(0, 1) la mediana simulada es {np.median(ingresos_mu1) / 1e6:5.2f} M€, "
      f"casi la misma")

# Buscamos una LogNormal que deje el percentil 95 de la contribución por debajo del 30 %.
print()
for roi_mu, roi_sigma in [(0.2, 0.9), (0.0, 0.5), (-0.2, 0.4), (-0.4, 0.3)]:
    _, aporte = simular_completo(roi_mu=roi_mu, roi_sigma=roi_sigma)
    print(f"roi ~ LogNormal({roi_mu:+.1f}, {roi_sigma:.1f}): "
          f"mediana {np.median(aporte):5.1f} %, p95 {np.percentile(aporte, 95):5.1f} %")

**Respuesta.** Antes de ver un solo dato, la priori le atribuye a la publicidad una mediana
del **25,7 %** de la facturación, y su percentil 95 se va al **62,3 %**. Es decir: le parece
perfectamente normal que seis de cada diez euros facturados vengan de los anuncios. Si eso
fuera verdad, no harían falta modelos.

En euros se ve mejor. El modelo simula una mediana de **10,94 M€** semanales contra los
**8,45 M€** observados: 2,48 M€ de más, que con una desviación típica semanal de 0,58 M€ son
**4,3 desviaciones típicas** de facturación inventada. Ese es exactamente el hueco que va a
tener que tapar `mu` en cuanto llegue el ajuste, y por eso su priori ha vuelto a
`Normal(0, 5)`.

Ojo aquí, que es la trampa de la trampa: si aprietas `mu` a `Normal(0, 1)` y vuelves a
simular, la previa **casi no se mueve** (10,79 M€ en vez de 10,94). La simulación previa no
detecta este error. Se ve después, en la posteriori, y lo compruebas en el ejercicio 5. Que
una comprobación no encuentre nada no significa que no haya nada.

Y para el 30 % del jefe: `LogNormal(0; 0,5)` deja la mediana en el 17,9 % y el percentil 95
en el **28,7 %**, justo por debajo. Fíjate en qué ha habido que apretar: la **sigma** de la
lognormal, de 0,9 a 0,5, más que su centro. El problema no era el ROI típico, era la cola.

## 5. Ajuste y diagnóstico del modelo completo

Catorce parámetros en vez de tres, y cinco de ellos —los $\alpha$— entran en el modelo de
una forma que los datos apenas distinguen. Esto ya no muestrea en dos segundos, pero
tampoco tarda tanto: mientras corre, piensa qué esperas ver.

In [ ]:
RUTA_IDATA = RAIZ / "notebooks" / "idata" / "modelo_meridian.nc"

# Ponlo a False si el muestreo se te hace largo y prefieres cargar el resultado guardado.
MUESTREAR = True

if MUESTREAR:
    with completo:
        idata = pm.sample(
            draws=1000, tune=1000, chains=4, target_accept=0.9, random_seed=SEMILLA
        )
    RUTA_IDATA.parent.mkdir(parents=True, exist_ok=True)
    idata.to_netcdf(RUTA_IDATA, groups=["posterior", "sample_stats", "observed_data"])
else:
    idata = az.from_netcdf(RUTA_IDATA)

with completo:  # la predictiva posterior se recalcula, que es rápido
    idata.extend(pm.sample_posterior_predictive(idata, random_seed=SEMILLA))

In [ ]:
resumen = az.summary(idata, var_names=["roi", "alpha", "gamma", "mu", "sigma"])
print("divergencias:", int(idata.sample_stats["diverging"].sum()))
print("r_hat máximo:", round(float(resumen["r_hat"].max()), 3))
print("ess_bulk mínimo:", int(resumen["ess_bulk"].min()))
resumen.round(2)

In [ ]:
az.plot_trace(idata, var_names=["roi", "sigma"], compact=True)
plt.tight_layout()
plt.show()

In [ ]:
az.plot_ppc(idata, num_pp_samples=100, colors=[ACENTO, "#7A7A7A", "black"])
plt.show()

### Ejercicio 5: diagnostica el modelo completo

- ¿Qué parámetro tiene el `ess` más bajo? ¿Te sorprende cuál es?
- Dibuja la posteriori de `alpha` de cada canal contra su priori uniforme. ¿En cuáles ha
  aprendido algo el modelo y en cuáles te está devolviendo la priori con otro nombre?
- Compara el `sigma` de este modelo con el del esqueleto. ¿Cuánto has ganado metiendo cinco
  canales de publicidad?
- Mira la posteriori de `mu`. Con la $\mathcal{N}(0, 1)$ que era razonable en el esqueleto,
  ¿habría podido llegar hasta ahí?
- La predictiva posterior reproduce bien el centro de la distribución. Busca dónde falla.

Ojo con la conclusión fácil: que un parámetro no se mueva de su priori **no es un fallo del
muestreador**. Es información. Significa que estos datos no distinguen entre un canal con
memoria de una semana y uno con memoria de un mes, y que lo que salga por el otro lado lo
estás poniendo tú.

In [ ]:
# Los tres parámetros peor muestreados.
print(resumen.sort_values("ess_bulk").head(3)[["mean", "sd", "ess_bulk", "r_hat"]].round(2))

# La priori de alpha es uniforme: su densidad es la recta y = 1.
alpha_post = idata.posterior["alpha"].values.reshape(-1, len(CANALES))

fig, ejes = plt.subplots(1, len(CANALES), figsize=(13, 2.8), sharey=True)
for i, (eje, canal) in enumerate(zip(ejes, CANALES)):
    eje.hist(alpha_post[:, i], bins=np.linspace(0, 1, 31), density=True,
             color=ACENTO, alpha=0.85, label="posteriori")
    eje.axhline(1.0, color="#AAAAAA", lw=2, label="priori")
    eje.set_title(canal, fontsize=10)
    eje.set_xlabel("α")
ejes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

# Lo que ha ganado el modelo metiendo cinco canales de publicidad.
sigma_completo = float(idata.posterior["sigma"].mean())
print(f"sigma esqueleto: {sigma_esqueleto:.3f}  ->  varianza explicada {1 - sigma_esqueleto ** 2:.1%}")
print(f"sigma completo:  {sigma_completo:.3f}  ->  varianza explicada {1 - sigma_completo ** 2:.1%}")

**Respuesta.** El `ess` más bajo es el de `mu`: 2.341. Y no es casualidad ni es un problema,
es el parámetro que se pasa el muestreo compensando lo que hacen los cinco canales, así que
va pegado a ellos y se mueve peor. Con `r_hat` 1,00 y cero divergencias, 2.341 muestras
efectivas de 4.000 es un ajuste sano.

Los `alpha` son el caso de libro. Las cinco posterioris salen con media entre 0,46 y 0,58 y
desviación típica entre 0,26 y 0,31. La uniforme en [0, 1] tiene media 0,50 y desviación
típica 0,29. O sea: el modelo no ha aprendido **nada** sobre ninguno de los cinco. Si en el
informe escribes "el efecto de `Channel0` dura unas tres semanas", ese tres lo has puesto tú
y no los datos.

Metiendo los cinco canales, `sigma` baja de 0,935 a 0,839: la varianza explicada pasa del
12,7 % al **29,6 %**. Es una mejora real, y sigue siendo un modelo que no explica el 70 % de
lo que ocurre.

Y `mu` se planta en **−2,76**. Con la `Normal(0, 1)` que era la correcta en el esqueleto,
eso está a casi tres desviaciones típicas del centro de su propia priori: el modelo habría
tenido que pelearse con ella para llegar, y lo habría pagado en divergencias y en `ess` por
los suelos. Ahí está la lección entera del taller: aquella priori no era mala, era mala
**para este otro modelo**.

La predictiva posterior, por su parte, clava el centro (desviación típica 1,02 contra 1,00,
percentil 5 idéntico) y falla por donde no esperabas: genera semanas de hasta −5
desviaciones típicas, que en euros es facturar menos de 6 M€, cuando la peor semana real fue
de 6,8 M€. No se queda corto, se pasa. Un error gaussiano no sabe que la facturación tiene
un suelo.

## 6. La pregunta que le importa a alguien

Nadie ha pedido nunca una posteriori. Piden a dónde va el dinero del trimestre que viene.

Antes de eso, el gráfico que resume el taller entero: priori contra posteriori del ROI de
cada canal. Lo que el modelo ha aprendido de los datos es la diferencia entre las dos.

In [ ]:
roi_previo = previa.prior["roi"].values.reshape(-1, len(CANALES))
roi_post = idata.posterior["roi"].values.reshape(-1, len(CANALES))

fig, ejes = plt.subplots(1, len(CANALES), figsize=(13, 3), sharey=True)
bordes = np.linspace(0, 12, 61)
for i, (eje, canal) in enumerate(zip(ejes, CANALES)):
    eje.hist(roi_previo[:, i], bins=bordes, density=True, color="#CCCCCC", label="priori")
    eje.hist(roi_post[:, i], bins=bordes, density=True, color=ACENTO, alpha=0.85,
             label="posteriori")
    eje.set_title(canal, fontsize=11)
    eje.set_xlim(0, 12)
    eje.set_xlabel("ROI")
ejes[0].legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
hdi = az.hdi(idata, var_names=["roi"], hdi_prob=0.9)["roi"].values
contrib = idata.posterior["contribucion"].values.reshape(-1, len(CANALES))

pd.DataFrame({
    "inversión (M€)": gasto_total / 1e6,
    "roi (mediana)": np.median(roi_post, axis=0),
    "roi hdi 90% bajo": hdi[:, 0],
    "roi hdi 90% alto": hdi[:, 1],
    "% ingresos (mediana)": 100 * np.median(contrib, axis=0) / y.sum(),
}, index=CANALES).round(2)

### Ejercicio 6: escribe la frase

Con las muestras de la posteriori, calcula:

- La probabilidad de que el canal con mejor ROI mediano sea de verdad mejor que el segundo:
  `(roi_post[:, i] > roi_post[:, j]).mean()`.
- La probabilidad de que cada canal esté perdiendo dinero (ROI < 1).
- El intervalo del 90 % del ingreso incremental de un canal, en euros.
- La contribución mediana de los cinco canales sumada, en porcentaje de la facturación.
  Compárala con lo que decía la priori en el ejercicio 4: ¿cuánto se ha movido?

Y luego escribe **una frase**, sin la palabra "posteriori" dentro, que puedas decirle a
quien firma el presupuesto.

In [ ]:
orden = np.argsort(np.median(roi_post, axis=0))[::-1]
mejor, segundo = orden[0], orden[1]

print(f"P({CANALES[mejor]} mejor que {CANALES[segundo]}) = "
      f"{(roi_post[:, mejor] > roi_post[:, segundo]).mean():.1%}")

print("\nprobabilidad de estar perdiendo dinero, P(ROI < 1):")
print(pd.Series((roi_post < 1).mean(axis=0), index=CANALES).round(2))

bajo, alto = np.percentile(contrib[:, mejor], [5, 95])
print(f"\ningreso incremental de {CANALES[mejor]}: 90 % entre {bajo / 1e6:.0f} y "
      f"{alto / 1e6:.0f} M€, con {gasto_total[mejor] / 1e6:.0f} M€ invertidos")

total_post = 100 * contrib.sum(axis=1) / y.sum()
print(f"\ncontribución de los medios: la priori decía {np.median(pct_previo):.1f} % "
      f"y la posteriori dice {np.median(total_post):.1f} %")

**Respuesta.** `Channel2` es el de mejor ROI mediano (1,84) y `Channel0` el segundo (1,31).
La probabilidad de que `Channel2` sea de verdad mejor que `Channel0` es del **63 %**. Poco
más que una moneda: con 12 M€ invertidos y tres canales que suben y bajan juntos, los datos
no dan para más.

La probabilidad de estar perdiendo dinero va del 0,23 de `Channel2` al 0,63 de `Channel4`. Y
mira `Channel3`: 0,53, prácticamente una moneda, y es el que se lleva 87,9 M€, cuatro de
cada diez euros del presupuesto. El ingreso incremental de `Channel2` está, con un 90 % de
probabilidad, entre 5 y 59 M€ sobre 12 M€ invertidos: un intervalo que lo mismo te describe
un canal ruinoso que uno excelente.

La contribución de los medios baja del 25,7 % que decía la priori al **18,9 %**. Se ha
movido, y hacia abajo: los datos, con lo poco que tienen que decir, dicen que la publicidad
aporta menos de lo que asumías. (Si sumas la columna de medianas de la tabla salen 17,3 %,
que no es lo mismo: la mediana de la suma no es la suma de las medianas.)

Y la frase, que es lo único que va a salir vivo de la reunión:

> Con estos datos, `Channel2` es el mejor candidato para subir presupuesto, pero no podemos
> afirmarlo: hay una probabilidad de una entre tres de que `Channel0` sea mejor. Lo que sí
> podemos decir es que `Channel3`, que se lleva cuatro de cada diez euros, tiene la mitad de
> probabilidades de estar perdiendo dinero, y que un experimento en ese canal nos daría más
> información que otro trimestre entero mirando el histórico.

## Para llevarte a casa

- Has ajustado dos modelos y en los dos la pregunta difícil fue la misma: qué priori y por
  qué. El código de pymc son cuatro líneas; todo lo demás es el oficio.
- Una priori no es de un parámetro, es del modelo entero. La misma $\mathcal{N}(0, 1)$ sobre
  $\mu$ era la correcta en el esqueleto y una restricción absurda en el modelo completo.
- Diagnosticar son dos preguntas, no una. El esqueleto muestreaba impecable y explicaba una
  miseria. Las dos cosas eran verdad a la vez.
- El modelo no ha descubierto los ROI: ha combinado unos datos flojos con una priori que
  has elegido tú. Cambia la priori y cambian los resultados. Eso no es un defecto del
  método bayesiano, es lo que el método te obliga a enseñar.
- La pregunta que hay que hacerle a cualquier MMM, propio o de proveedor, es la del
  ejercicio 4: **¿qué contribución de medios asume tu priori antes de ver los datos?** Si no
  saben responder, el número que te van a dar es el suyo, no el de los datos.
- Lo que dejamos fuera: la saturación —la curva de Hill, que es la que dice cuándo dejar de
  invertir en un canal—, el canal orgánico, la variable `Promo`, la estacionalidad, la
  calibración del ROI con experimentos y, claro, el modelo jerárquico por regiones, que es
  donde de verdad brilla Meridian.